In [1]:
import os
import warnings
import pandas as pd

from dotenv import load_dotenv
from gliner import GLiNER
from qdrant_client import QdrantClient
from neo4j import GraphDatabase
from fastembed import TextEmbedding

warnings.filterwarnings("ignore")

In [2]:
#load environment variables

load_dotenv()

True

In [3]:
#qdrant configuration

QDRANT_URL=os.getenv(
    "QDRANT_URL"
)

QDRANT_API_KEY=os.getenv(
    "QDRANT_API_KEY"
)

COLLECTION_NAME="healthcare_conversations"

In [4]:
#neo4j configuration

NEO4J_URI=os.getenv(
    "NEO4J_URI"
)

NEO4J_USERNAME=os.getenv(
    "NEO4J_USERNAME"
)

NEO4J_PASSWORD=os.getenv(
    "NEO4J_PASSWORD"
)

In [5]:
#initialize qdrant client

qdrant_client=QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY
)

print("qdrant connected")

qdrant connected


In [6]:
#initialize neo4j driver

neo4j_driver=GraphDatabase.driver(
    NEO4J_URI,
    auth=(
        NEO4J_USERNAME,
        NEO4J_PASSWORD
    )
)

print("neo4j connected")

neo4j connected


In [7]:
#initialize embedding model

embedding_model=TextEmbedding(
    model_name="BAAI/bge-small-en-v1.5"
)

print("embedding model loaded")

embedding model loaded


In [8]:
#initialize gliner model

gliner_model=GLiNER.from_pretrained(
    "urchade/gliner_medium-v2.1"
)

print("gliner model loaded")

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

gliner model loaded


In [9]:
#define medical labels

labels=[
    "symptom",
    "disease",
    "medication",
    "medical_condition",
    "body_part"
]

In [10]:
#extract medical entities

def extract_medical_entities(query):

    entities=gliner_model.predict_entities(
        query,
        labels
    )

    extracted=[]

    for entity in entities:

        extracted.append({

            "text":entity["text"].lower(),
            "label":entity["label"]

        })

    return extracted

In [11]:
#generate embeddings

def generate_embedding(text):

    embedding=list(

        embedding_model.embed(
            [text]
        )

    )[0]

    return embedding

In [20]:
#semantic retrieval from qdrant

def qdrant_retrieval(query):

    query_vector=generate_embedding(
        query
    )

    search_results=qdrant_client.query_points(

        collection_name=COLLECTION_NAME,

        query=query_vector,

        limit=5

    ).points

    retrieved_contexts=[]

    for result in search_results:

        retrieved_contexts.append({

            "score":result.score,

            "context":result.payload[
                "retrieval_text"
            ]

        })

    return retrieved_contexts

In [21]:
#graph retrieval from neo4j

def neo4j_graph_retrieval(entities):

    graph_contexts=[]

    with neo4j_driver.session() as session:

        for entity in entities:

            entity_name=entity["text"]

            result=session.run("""

            MATCH (a)-[r]->(b)

            WHERE toLower(a.name)=toLower($entity)

            RETURN a.name AS source,
                   type(r) AS relationship,
                   b.name AS target

            LIMIT 10

            """,

            entity=entity_name

            )

            for record in result:

                graph_contexts.append({

                    "source":record["source"],
                    "relationship":record["relationship"],
                    "target":record["target"]

                })

    return graph_contexts

In [22]:
#hybrid retrieval pipeline

def hybrid_retrieval_pipeline(user_query):

    print("\nuser query\n")
    print(user_query)

    #entity extraction

    extracted_entities=extract_medical_entities(
        user_query
    )

    print("\nextracted entities\n")
    print(extracted_entities)

    #semantic retrieval

    semantic_contexts=qdrant_retrieval(
        user_query
    )

    print("\nsemantic retrieval completed")

    #graph retrieval

    graph_contexts=neo4j_graph_retrieval(
        extracted_entities
    )

    print("\ngraph retrieval completed")

    #final hybrid context

    hybrid_context={

        "query":user_query,

        "entities":extracted_entities,

        "semantic_contexts":semantic_contexts,

        "graph_contexts":graph_contexts

    }

    return hybrid_context

In [23]:
#test healthcare query

query="""
I have chest pain,dizziness and breathing difficulty
"""

hybrid_result=hybrid_retrieval_pipeline(
    query
)


user query


I have chest pain,dizziness and breathing difficulty


extracted entities

[{'text': 'chest pain', 'label': 'symptom'}, {'text': 'dizziness', 'label': 'symptom'}, {'text': 'breathing difficulty', 'label': 'symptom'}]

semantic retrieval completed

graph retrieval completed


In [24]:
#print extracted entities

print("\nmedical entities\n")

for entity in hybrid_result["entities"]:

    print(entity)


medical entities

{'text': 'chest pain', 'label': 'symptom'}
{'text': 'dizziness', 'label': 'symptom'}
{'text': 'breathing difficulty', 'label': 'symptom'}


In [25]:
#print semantic retrieval

print("\nsemantic contexts\n")

for context in hybrid_result[
    "semantic_contexts"
]:

    print(context)

    print("\n")


semantic contexts

{'score': 0.7659187, 'context': '\nPatient Query:\nHi hope you can helpive been getting chest pains, not severe but a constant pain, can happen when im just resting but has also happened when im exercising, it continues through to my back as well. I sometimes wake up with chest pains too. i have a strong history of heart disease. grandfather died at 26 from a heat attack and mother had one at 45, also grandmother on the other side had a heart attack. Could you please explain what is happening? Thanks\n\nDoctor Response:\nHelloThanks for posting at Chat Doctor. You have not provided your age on the details. I would have been able to help u better had you mentioned your age. You have complaints of chest pain radiating to back with strong family history. So in your case it is better that we evaluate you for presence of heart disease. I recommend an ECG and 2d echo initially. If both these tests are normal you can proceed for a treadmill test which requires you to walk 

In [28]:
#print graph retrieval

print("\ngraph contexts\n")

for context in hybrid_result[
    "graph_contexts"
]:

    print(context)


graph contexts

{'source': 'chest pain', 'relationship': 'RELATED_TO', 'target': 'bronchitis'}
{'source': 'chest pain', 'relationship': 'RELATED_TO', 'target': 'gallstones'}
{'source': 'chest pain', 'relationship': 'RELATED_TO', 'target': 'asthma'}
{'source': 'chest pain', 'relationship': 'RELATED_TO', 'target': 'uri'}
{'source': 'chest pain', 'relationship': 'RELATED_TO', 'target': 'sinusitis'}
{'source': 'chest pain', 'relationship': 'RELATED_TO', 'target': 'copd'}
{'source': 'chest pain', 'relationship': 'RELATED_TO', 'target': 'lyme disease'}
{'source': 'chest pain', 'relationship': 'RELATED_TO', 'target': 'acute gastritis'}
{'source': 'chest pain', 'relationship': 'RELATED_TO', 'target': 'wolffparkinsonwhite syndrome'}
{'source': 'chest pain', 'relationship': 'RELATED_TO', 'target': 'chest pain'}
{'source': 'dizziness', 'relationship': 'RELATED_TO', 'target': 'osteoporosis'}
{'source': 'dizziness', 'relationship': 'RELATED_TO', 'target': 'cancer'}
{'source': 'dizziness', 'relatio

In [29]:
#create final hybrid medical context

final_context=[]

for semantic in hybrid_result[
    "semantic_contexts"
]:

    final_context.append(

        semantic["context"]

    )

for graph in hybrid_result[
    "graph_contexts"
]:

    graph_text=f"""

    {graph['source']}
    {graph['relationship']}
    {graph['target']}

    """

    final_context.append(
        graph_text
    )

print("\nfinal hybrid context\n")

for item in final_context:

    print(item)

    print("\n")


final hybrid context


Patient Query:
Hi hope you can helpive been getting chest pains, not severe but a constant pain, can happen when im just resting but has also happened when im exercising, it continues through to my back as well. I sometimes wake up with chest pains too. i have a strong history of heart disease. grandfather died at 26 from a heat attack and mother had one at 45, also grandmother on the other side had a heart attack. Could you please explain what is happening? Thanks

Doctor Response:
HelloThanks for posting at Chat Doctor. You have not provided your age on the details. I would have been able to help u better had you mentioned your age. You have complaints of chest pain radiating to back with strong family history. So in your case it is better that we evaluate you for presence of heart disease. I recommend an ECG and 2d echo initially. If both these tests are normal you can proceed for a treadmill test which requires you to walk on a treadmill while your ECG is co

In [31]:
#close neo4j connection

neo4j_driver.close()

print("neo4j connection closed")

neo4j connection closed
